# Experimenting with Delta Lakes

## Dev Jeison Robles Arias


## Creating a table, defining schema (explicit) and creating a Spark DataFrame from data set

In [0]:
# Simple DataSet
data = [
    (1, "Alice", 100),
    (2, "Bob", 200)
    ]

# Creating spark DataFrame with explict schema
df = spark.createDataFrame(data, ["id","name","amount"])

# Savving that DF into a Delta Lake table
df.write.format("delta").mode("overwrite").saveAsTable("delta_demo_table")

## Loading Table from Delta Lake

Will explore modes: '''overwrite''' and '''append'''

In [0]:
# Reading the Dalta Lake table and creating a new Spark Data Frame
delta_df = spark.table("delta_demo_table")
display(delta_df)

___
## Exploring TimeTravel (what makes it different from common datalakes)

In [0]:
data2 = [
    (3, "Charlie", 300),
    (4, "Dave", 400)
    ]

# Creating spark DataFrame with explict schema
df2 = spark.createDataFrame(data2, ["id","name","amount"])

# Adding to the Delta Lake table
df2.write.format("delta").mode("append").saveAsTable("delta_demo_table")

In [0]:

# Exploring versions
print("Version 0:")
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("delta_demo_table")
display(df_v0)


print("\nVersion 1:")
df_v1 = spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .table("delta_demo_table")
display(df_v1)

## Schema Enforcement

Lets try to append a new register but with a different explicit schema.

An error should happend because Delta Lakes evaluate consistency. Delta Lakes follows ACID operations.

A -> Atomicy
C -> Consistency (this one is fundamental here)
I -> Isolation
D -> Durability

In another datalakes use csv or parquet files ```This Op is available in that ones```

In [0]:
# Define data
data3 = [(5, "David", 400, "CR")] # Note is another structure (adding country)
df3 = spark.createDataFrame(data3, ["id","name","amount","country"])

df3.write.format("delta").mode("append").saveAsTable("delta_demo_table") 

### Schema Evolution

In [0]:
# Using mergeSchema into the options allows to add new columns
df3.write.format("delta") \
   .mode("append") \
   .option("mergeSchema", "true")\
   .saveAsTable("delta_demo_table")

Lets go to inspect before and after schema evolve with time travel

In [0]:
print("Inpecting before evolving schema....")
bef = spark.read.format("delta").option("versionAsOf",1).table("delta_demo_table")
display(bef)

print("\nInpecting after evolving schema....")
aft = spark.read.format("delta").option("versionAsOf",2).table("delta_demo_table")
display(aft)


## Upsert (Update + Insert)

This is a process for updating existing records or inserting new ones in a Delta table. This is especially useful for synchronizing datasets or pipelines, where you want to add new rows and update existing ones in a single atomic operation.

In Databricks Delta Lake, this is performed using the MERGE command or its PySpark interface (DeltaTable.merge). The upsert logic works by:

Matching rows with a given key (e.g., id in your notebook) between the source (updates) and the target table.
If the row exists in the Delta table, it’s updated.
If the row doesn’t exist, a new row is inserted.


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "delta_demo_table")

updates = spark.createDataFrame(
    [(1, "Alice", 150)],
    ["id","name", "amount"])
delta_table.alias("t")\
    .merge(
        updates.alias("u"),
        "t.id = u.id"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll() \
    .execute()

## Rolling back to previus state

The avove error happened because an schema difference. In the next code we will '''roll back''' to a previus version where the country wasn't inserted yet.

In [0]:
# Roll back to version 1 (before 'country' field existed)
df_rollback = spark.read.format("delta").option("versionAsOf", 1).table("delta_demo_table")
display(df_rollback)

In [0]:
# Jump back to version 1 and overwrite the table
df_rollback = spark.read.format("delta").option("versionAsOf", 1).table("delta_demo_table")
df_rollback.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("delta_demo_table")

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "delta_demo_table")

updates = spark.createDataFrame(
    [(1, "Alice", 150)],
    ["id","name", "amount"])

delta_table.alias("t")\
    .merge(
        updates.alias("u"),
        "t.id = u.id"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll() \
    .execute()

# Display the table after upsert
display(spark.table("delta_demo_table"))